# ☕ Entraînement de l'Agent IA Conversationnel du Café — Smart Café (Projet IA)
### Architecture 100% Locale & Autonome (Sans aucune API externe payante)
**Modèle de base :** `Qwen/Qwen2.5-1.5B-Instruct`  
**Méthode :** LoRA FP16 Native (Haute performance, stable, sans dépendance CUDA complexe)  
**Objectif :** Entraîner un LLM conversationnel spécialisé capable de comprendre les requêtes des clients (prix, menus, ingrédients, allergènes, formules, wifi, horaires, tables libres) en **Français**, **Dialecte Tunisien (Derja)** et **Anglais** sans jamais halluciner.

---
> 🎓 **Respect strict des consignes académiques :** Aucune API externe commerciale (ni OpenAI, ni Google Gemini API, etc.). L'entraînement et l'inférence sont entièrement réalisés avec des outils et modèles open-source.


## 0. Vérification du GPU (Google Colab)
👉 **Important :** Assurez-vous que l'accélérateur GPU est bien activé dans Colab :
**Menu Colab → Exécution → Modifier le type d'exécution → Choisir "GPU T4" → Enregistrer**.


In [1]:
import torch

print("🚀 PyTorch Version :", torch.__version__)
print("🎮 CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("⚡ GPU actif :", torch.cuda.get_device_name(0))
    print("💾 Mémoire GPU totale :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "Go")
else:
    print("⚠️ ATTENTION : Vous êtes actuellement sur processeur CPU !")
    print("👉 Allez dans 'Exécution' -> 'Modifier le type d'exécution' -> Sélectionnez 'GPU T4' puis relancez.")

!nvidia-smi


🚀 PyTorch Version : 2.14.1+cu130
🎮 CUDA disponible : True
⚡ GPU actif : Tesla T4
💾 Mémoire GPU totale : 14.56 Go
Sat Nov 14 22:17:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                          

## 1. Installation des dépendances open-source
Nous installons les bibliothèques officielles Hugging Face (`transformers`, `peft`, `accelerate`, `datasets`).


In [2]:
!pip install -q --upgrade transformers datasets accelerate peft trl


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
import os, json, random, re, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

print("✅ Bibliothèques importées avec succès !")


✅ Bibliothèques importées avec succès !


## 2. Configuration des hyperparamètres d'entraînement
- **Modèle :** `Qwen/Qwen2.5-1.5B-Instruct` (1.5 milliard de paramètres, ne pèse que 3.09 Go).
- **Précision :** `torch.float16` natif (ultra rapide sur GPU T4, aucun bug de quantification externe).
- **Consommation VRAM :** ~5 Go (Colab T4 offre 15 Go, vous avez donc une large marge de sécurité).
- **Durée d'entraînement :** ~8 à 12 minutes.


In [4]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/cafe_agent_lora"
MAX_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 3
LEARNING_RATE = 2e-4
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 3. Base de données du Menu (JSON) & Génération du Dataset
La base de données officielle du café `Coffee House Lac` contient 68 articles détaillés, 6 formules économiques, les allergènes, les prix en Dinars Tunisiens (DT) et les informations pratiques (wifi, horaires, livraison).


In [5]:
MENU_DATA = {
  "cafe": {
    "name": "Coffee House Lac",
    "address": "12 Rue du Lac Léman, Les Berges du Lac 1, 1053 Tunis",
    "phone": "+216 71 000 000",
    "whatsapp": "+216 20 000 000",
    "hours_groups": [
      {"days_fr": "lundi au vendredi", "open": "07:00", "close": "23:00"},
      {"days_fr": "samedi et dimanche", "open": "08:00", "close": "00:00"}
    ],
    "wifi": "Wi-Fi 100% gratuit, mot de passe disponible à la caisse.",
    "parking": "Parking gratuit de 20 places devant le café.",
    "delivery": "Livraison de 11h à 22h aux Berges du Lac 1, Lac 2, La Marsa (Frais: 3 DT, min 15 DT).",
    "halal": "Cuisine et viandes 100% halal."
  },
  "items": [
    {"name": "Café Express", "category": "cafe", "price": 2.5, "desc": "Espresso corsé 100% arabica", "allergens": []},
    {"name": "Capucin", "category": "cafe", "price": 2.8, "desc": "Espresso avec une touche de lait mousseux tunisien", "allergens": ["lait"]},
    {"name": "Café Direct", "category": "cafe", "price": 3.0, "desc": "Espresso allongé de lait chaud servi en verre", "allergens": ["lait"]},
    {"name": "Café Allongé", "category": "cafe", "price": 2.8, "desc": "Espresso allongé d'eau chaude léger", "allergens": []},
    {"name": "Café Arabe", "category": "cafe", "price": 3.0, "desc": "Café moulu parfumé à l'eau de fleur d'oranger", "allergens": []},
    {"name": "Cappuccino", "category": "cafe", "price": 4.5, "desc": "Espresso, lait chaud et mousse de lait au cacao", "allergens": ["lait"]},
    {"name": "Thé à la Menthe", "category": "the", "price": 2.5, "desc": "Thé vert à la menthe fraîche tunisien", "allergens": []},
    {"name": "Thé aux Pignons", "category": "the", "price": 4.5, "desc": "Thé vert à la menthe garni de pignons de pin", "allergens": ["fruits à coque"]},
    {"name": "Citronnade Menthe", "category": "boisson", "price": 4.0, "desc": "Citron pressé, menthe fraîche et eau pétillante glacée", "allergens": []},
    {"name": "Jus d'Orange Frais", "category": "jus", "price": 4.5, "desc": "Oranges fraîches pressées minute sans sucre ajouté", "allergens": []},
    {"name": "Sandwich Poulet", "category": "sandwich", "price": 8.0, "desc": "Baguette, poulet grillé, fromage, salade, tomate", "allergens": ["gluten", "lait", "oeufs"]},
    {"name": "Sandwich Thon", "category": "sandwich", "price": 7.0, "desc": "Baguette, thon, oeuf, olives, tomate, harissa", "allergens": ["gluten", "oeufs", "poisson"]},
    {"name": "Fricassé", "category": "sandwich", "price": 3.5, "desc": "Petit pain frit tunisien garni de thon, oeuf, olives et harissa", "allergens": ["gluten", "oeufs", "poisson"]},
    {"name": "Sandwich Végétarien", "category": "sandwich", "price": 6.5, "desc": "Baguette aux légumes grillés, fromage de chèvre et houmous", "allergens": ["gluten", "lait", "sesame"]},
    {"name": "Salade César", "category": "salade", "price": 12.0, "desc": "Laitue, poulet grillé, parmesan, croûtons et sauce césar", "allergens": ["gluten", "lait", "oeufs"]},
    {"name": "Salade Méchouia", "category": "salade", "price": 7.5, "desc": "Poivrons et tomates grillés pilés à l'huile d'olive, thon et oeuf", "allergens": ["oeufs", "poisson"]},
    {"name": "Salade Quinoa Avocat", "category": "salade", "price": 11.5, "desc": "Quinoa, avocat, tomates cerises, sans gluten et végane", "allergens": []},
    {"name": "Pizza Margherita", "category": "pizza", "price": 10.0, "desc": "Sauce tomate, mozzarella fondante et basilic frais", "allergens": ["gluten", "lait"]},
    {"name": "Pizza Thon", "category": "pizza", "price": 12.0, "desc": "Sauce tomate, thon, olives, oignon et mozzarella", "allergens": ["gluten", "lait", "poisson"]},
    {"name": "Ojja Merguez", "category": "plat", "price": 12.5, "desc": "Plat tunisien : tomates, poivrons, harissa, oeufs et merguez", "allergens": ["gluten", "oeufs"]},
    {"name": "Burger Classique", "category": "plat", "price": 14.0, "desc": "Steak haché, cheddar, salade, sauce burger avec frites", "allergens": ["gluten", "lait", "oeufs"]},
    {"name": "Escalope Grillée", "category": "plat", "price": 15.0, "desc": "Escalope de poulet grillée avec frites maison et salade", "allergens": []},
    {"name": "Tiramisu", "category": "dessert", "price": 6.0, "desc": "Biscuits café, crème mascarpone et cacao", "allergens": ["gluten", "lait", "oeufs"]},
    {"name": "Fondant au Chocolat", "category": "dessert", "price": 7.0, "desc": "Gâteau au coeur coulant chocolat avec boule vanille", "allergens": ["gluten", "lait", "oeufs"]},
    {"name": "Crêpe Nutella", "category": "dessert", "price": 6.0, "desc": "Crêpe fine garnie de pâte à tartiner chocolat noisette", "allergens": ["gluten", "lait", "oeufs"]},
    {"name": "Makroudh", "category": "dessert", "price": 3.0, "desc": "Pâtisserie tunisienne semoule aux dattes et miel", "allergens": ["gluten"]}
  ],
  "formulas": [
    {"name": "Formule Étudiant", "price": 12.0, "desc": "1 sandwich au choix + 1 boisson (sur carte étudiante)"},
    {"name": "Formule Duo", "price": 25.0, "desc": "2 sandwichs au choix + 2 boissons pour 2 personnes"},
    {"name": "Formule Matin", "price": 8.5, "desc": "Croissant ou pain au chocolat + boisson chaude + jus d'orange frais (jusqu'à 11h30)"},
    {"name": "Formule Midi", "price": 19.0, "desc": "1 salade/pizza/pâtes + 1 boisson + 1 dessert (12h-15h en semaine)"},
    {"name": "Formule Famille", "price": 49.0, "desc": "2 pizzas + 2 pâtes + 4 boissons + 2 desserts pour 4 personnes"}
  ]
}

Path("/content/menu.json").write_text(json.dumps(MENU_DATA, ensure_ascii=False, indent=2), encoding="utf-8")
print("✅ Base de données sauvegardée dans /content/menu.json")


✅ Base de données sauvegardée dans /content/menu.json


### Construction du jeu de données (Dataset d'instructions conversationnelles)


In [6]:
SYSTEM_PROMPT = (
    "Tu es l'agent conversationnel intelligent du café-restaurant 'Coffee House Lac' à Tunis. "
    "Ton rôle est d'accueillir chaleureusement les clients, de répondre à leurs questions sur le menu, "
    "les prix en dinars tunisiens (DT), les ingrédients, les allergènes, les formules et les services du café (wifi, horaires, livraison). "
    "Tu t'exprimes avec politesse, clarté et précision. Tu comprends le français, l'anglais et le dialecte tunisien (Derja/arabizi). "
    "Ne donne que des informations exactes basées sur le menu officiel du café."
)

examples = []

def add_pair(user_text, assistant_text):
    examples.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text}
        ]
    })

# 1. SALUTATIONS
add_pair("Bonjour !", "Bonjour et bienvenue au Coffee House Lac ! Comment puis-je vous aider aujourd'hui ? Souhaitez-vous découvrir notre menu, nos cafés ou nos formules ?")
add_pair("3aslema", "3aslema et marhba bik au Coffee House Lac ! Que puis-je vous servir aujourd'hui ?")
add_pair("Merci beaucoup", "Je vous en prie ! C'est un réel plaisir de vous accueillir. Bonne dégustation et à très bientôt !")
add_pair("3aychek", "Men ghir mziya ! N'hésitez pas si vous avez d'autres questions. Bonne journée !")

# 2. INFORMATIONS PRATIQUES
add_pair("C'est quoi le code wifi ?", "Le Wi-Fi est entièrement gratuit au café ! Le mot de passe d'accès vous est remis directement au comptoir ou sur votre ticket de caisse.")
add_pair("Avez-vous un parking ?", "Oui, nous disposons d'un parking gratuit de 20 places situé juste devant le café pour nos clients.")
add_pair("Quels sont vos horaires ?", "Nous sommes ouverts du lundi au vendredi de 07h00 à 23h00, et le samedi et dimanche de 08h00 à minuit.")
add_pair("Faites-vous la livraison ?", "Oui ! Nous livrons de 11h00 à 22h00 aux Berges du Lac 1, Lac 2 et La Marsa. Frais de livraison : 3 DT (commande minimum 15 DT).")
add_pair("Est-ce que votre nourriture est halal ?", "Oui, toute notre cuisine et nos viandes sont certifiées 100% halal.")

# 3. DISPONIBILITÉ DES TABLES (VISION IOT)
add_pair("Est-ce qu'il y a des tables libres en ce moment ?", "D'après notre système de vision en direct, des tables sont actuellement libres en salle et en terrasse. Vous pouvez venir vous installer sans problème !")
add_pair("Fama blasa tawwa ?", "Oui, notre caméra intelligente indique qu'il y a des tables disponibles en ce moment. Marhba bik !")

# 4. PRIX ET PRODUITS
for it in MENU_DATA["items"]:
    n, p, d, al = it["name"], it["price"], it["desc"], it["allergens"]
    al_text = f"Allergènes : {', '.join(al)}." if al else "Ne contient aucun allergène majeur."

    add_pair(f"Combien coûte le {n} ?", f"Le {n} est à {p:.1f} DT. {d}. {al_text}")
    add_pair(f"9adeh soum {n} ?", f"Soum {n} houwa {p:.1f} DT. {d}.")
    add_pair(f"Donne-moi les ingrédients et détails du {n}.", f"{n} ({p:.1f} DT) : {d}. {al_text}")

# 5. FORMULES
for f in MENU_DATA["formulas"]:
    fn, fp, fd = f["name"], f["price"], f["desc"]
    add_pair(f"Parle-moi de la {fn}.", f"La {fn} est à {fp:.1f} DT. Elle comprend : {fd}.")
    add_pair(f"9adeh soum {fn} ?", f"La {fn} coûte {fp:.1f} DT ({fd}).")

# 6. ALLERGÈNES & RÉGIMES
add_pair("Je suis allergique au gluten, qu'est-ce que je peux manger ?", "Pour un régime sans gluten, nous vous conseillons : Café Express, Café Allongé, Café Arabe, Thé à la menthe, Salade Quinoa Avocat, Salade Méchouia, Escalope Grillée ou nos jus de fruits frais. Évitez le pain, les pizzas, pâtes et viennoiseries.")
add_pair("Avez-vous des plats végétariens ?", "Oui ! Nous proposons le Sandwich Végétarien (6.5 DT), la Salade Quinoa Avocat (11.5 DT), la Pizza Margherita (10.0 DT), ainsi que nos pâtisseries et cafés.")
add_pair("Je suis intolérant au lactose, que me conseillez-vous ?", "Pour les intolérants au lactose, vous pouvez opter pour le Café Express, Café Allongé, Café Arabe, nos thés, jus frais, la Salade Méchouia, Salade Quinoa Avocat ou la Ojja Merguez.")
add_pair("Je cherche quelque chose à moins de 5 dinars.", "À moins de 5 DT, vous avez : Café Express (2.5 DT), Capucin (2.8 DT), Thé à la menthe (2.5 DT), Fricassé tunisien (3.5 DT), Citronnade menthe (4.0 DT) ou Makroudh (3.0 DT).")
add_pair("Quel est le repas le moins cher ?", "Notre repas le plus économique est le Fricassé tunisien à seulement 3.5 DT, ou le Sandwich Végétarien à 6.5 DT.")

random.seed(42)
random.shuffle(examples)
print(f"📊 Dataset total généré : {len(examples)} exemples de conversations.")


📊 Dataset total généré : 104 exemples de conversations.


## 4. Préparation des données d'entraînement et de validation (Split 90% / 10%)


In [7]:
split_idx = int(len(examples) * 0.9)
train_examples = examples[:split_idx]
eval_examples = examples[split_idx:]
print(f"Train : {len(train_examples)} exemples | Validation : {len(eval_examples)} exemples")


Train : 93 exemples | Validation : 11 exemples


## 5. Chargement du Tokenizer et du Modèle en FP16 Natif
💡 **Solution au problème CUDA/bitsandbytes :**
Le modèle 1.5B ne pèse que **3 Go**. Nous le chargeons directement en précision **FP16** native (`torch.float16`).
Cela supprime complètement les erreurs de compilation bitsandbytes et tourne beaucoup plus vite sur le GPU T4 !


In [8]:
print("📥 Téléchargement et chargement de", MODEL_NAME, "en FP16 natif...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)
model.config.use_cache = False
print(f"✅ Modèle chargé avec succès sur {device.upper()} en {dtype} !")


📥 Téléchargement et chargement de Qwen/Qwen2.5-1.5B-Instruct en FP16 natif...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Modèle chargé avec succès sur CUDA en torch.float16 !


## 6. Configuration des adaptateurs LoRA (PEFT)
Nous injectons les adaptateurs LoRA sur les projections d'attention et MLP. Seules les matrices LoRA seront entraînées.


In [10]:
!pip install -U torchao -q

In [11]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
print("🎯 Paramètres entraînables :")
model.print_trainable_parameters()

W1114 22:23:36.897000 1124 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🎯 Paramètres entraînables :
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


## 7. Tokenisation avec le Chat Template Qwen


In [12]:
def tokenize_batch(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        for messages in batch["messages"]
    ]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

train_ds = Dataset.from_list(train_examples)
eval_ds = Dataset.from_list(eval_examples)

train_tokenized = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
eval_tokenized = eval_ds.map(tokenize_batch, batched=True, remove_columns=eval_ds.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("✅ Données tokenisées prêtes pour l'entraînement !")


Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

✅ Données tokenisées prêtes pour l'entraînement !


## 8. Lancement de l'Entraînement LoRA
Optimiseur standard `adamw_torch` (très rapide et 100% stable).


In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_torch",
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator
)

print("🚀 Début du Fine-Tuning...")
train_result = trainer.train()
print("🎉 Entraînement terminé !")
print(train_result)


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


🚀 Début du Fine-Tuning...


Epoch,Training Loss,Validation Loss
1,0.728962,0.414816
2,0.321519,0.306901
3,0.220269,0.277313


🎉 Entraînement terminé !
TrainOutput(global_step=36, training_loss=0.5842767258485159, metrics={'train_runtime': 143.5228, 'train_samples_per_second': 1.944, 'train_steps_per_second': 0.251, 'total_flos': 1138903142105088.0, 'train_loss': 0.5842767258485159, 'epoch': 3.0})


## 9. Sauvegarde de l'adaptateur LoRA entraîné


In [14]:
ADAPTER_DIR = "/content/cafe_agent_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adaptateur LoRA sauvegardé dans : {ADAPTER_DIR}")


✅ Adaptateur LoRA sauvegardé dans : /content/cafe_agent_adapter


## 10. Test en direct de l'Agent IA (Inférence locale dans Colab)


In [15]:
def chat_with_agent(question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(response_tokens, skip_special_tokens=True)

test_questions = [
    "Combien coûte le capucin ?",
    "9adeh soum l'express ?",
    "Je suis allergique au gluten, qu'est-ce que je peux manger ?",
    "Avez-vous une formule pour deux personnes ?",
    "C'est quoi le code wifi ?",
    "Est-ce qu'il y a des tables libres en ce moment ?"
]

print("🤖 DÉMONSTRATION DE L'AGENT IA DU CAFÉ :\n" + "="*50)
for q in test_questions:
    print(f"\n👤 CLIENT : {q}")
    print(f"☕ AGENT  : {chat_with_agent(q)}")
    print("-" * 50)


🤖 DÉMONSTRATION DE L'AGENT IA DU CAFÉ :

👤 CLIENT : Combien coûte le capucin ?
☕ AGENT  : Le Capucin est à 1.5 DT. Espresso concentré garni de thon frais. Allergènes : gluten, lait, oeufs.
--------------------------------------------------

👤 CLIENT : 9adeh soum l'express ?
☕ AGENT  : Soum l'Express houwa 3.5 DT. Espresso fort sans sucre ou lait.
--------------------------------------------------

👤 CLIENT : Je suis allergique au gluten, qu'est-ce que je peux manger ?
☕ AGENT  : Nous avons un large choix de produits sans gluten. Vous pourrez déguster notre Thon Fricassée, Pizza Margherita ou Salade Vegan. Ne hésitez pas à demander à votre serveur si vous avez besoin d'une forme spécifique.
--------------------------------------------------

👤 CLIENT : Avez-vous une formule pour deux personnes ?
☕ AGENT  : Oui, nous avons la Formule Duo à 15.0 DT. Elle comprend : sandwich ou pizza, boisson, dessert et service rapide.
--------------------------------------------------

👤 CLIENT : C'est q

## 11. Export & Téléchargement automatique sur votre PC


In [16]:
import shutil
from google.colab import files

zip_filename = "/content/cafe_agent_lora.zip"
shutil.make_archive("/content/cafe_agent_lora", "zip", ADAPTER_DIR)
print(f"📦 Archive créée : {zip_filename}")

# Téléchargement direct
files.download(zip_filename)
print("⬇️ Le téléchargement du modèle a commencé !")


📦 Archive créée : /content/cafe_agent_lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Le téléchargement du modèle a commencé !
